<a href="https://colab.research.google.com/github/vv2281/MAT-422/blob/main/HW2.4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, stats

rng = np.random.default_rng(422)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

#2.4.1 MLE for Random Samples

##What is the likelihood?
Suppose X₁, X₂, …, Xₙ are a random sample: independent draws from the same distribution, f(x; θ), where θ is a parameter we don't know.

Once we have the data, we can ask: for each possible value of θ, how likely is the data we actually saw? That function of θ is the likelihood: L(θ) = f(x₁; θ) · f(x₂; θ) · … · f(xₙ; θ)

We can multiply because the observations are independent.

The maximum likelihood estimate (MLE) is the value of θ that makes L(θ) as large as possible. In words: it is the parameter value that best explains the data we observed.

##Why we use the log-likelihood
Instead of maximizing L(θ), we maximize its log:  ℓ(θ) = log L(θ) = log f(x₁; θ) + … + log f(xₙ; θ)

This gives the same answer because log always increases. It is easier because sums are easier to take derivatives of than products. It also avoids a computer problem: multiplying many small numbers together rounds down to 0.

Steps for finding an MLE:
1. Write down ℓ(θ).
2. Take the derivative and set it equal to 0.
3. Solve for θ.
4. Check the second derivative is negative, so it is a maximum.

In [ ]:
#product of 2000 densities underflows to 0. the sum of logs does not
x_demo = rng.normal(0, 1, size=2000)
print("Likelihood (product):    ", np.prod(stats.norm.pdf(x_demo)))
print("Log-likelihood (sum):    ", round(np.sum(stats.norm.logpdf(x_demo)), 2))

##Example 1: Bernoulli

Xᵢ is 0 or 1 with P(Xᵢ = 1) = p. Let s = number of 1s.  ℓ(p) = s·log(p) + (n − s)·log(1 − p),
 ℓ'(p) = s/p − (n − s)/(1 − p) = 0   →   p̂ = s/n

ℓ''(p) < 0, so this is a maximum. The MLE is the sample proportion.

In [ ]:
p_true, n = 0.3, 50
x_bern = rng.binomial(1, p_true, size=n)
s = x_bern.sum()

p_grid = np.linspace(0.01, 0.99, 500)
ll = s * np.log(p_grid) + (n - s) * np.log(1 - p_grid)
p_hat = s / n

print(f"s = {s}, n = {n}")
print(f"Formula  p_hat = s/n = {p_hat:.4f}")
print(f"Grid-search maximum  = {p_grid[np.argmax(ll)]:.4f}")

plt.plot(p_grid, ll)
plt.axvline(p_hat, color="red", ls="--", label=f"MLE = {p_hat:.2f}")
plt.axvline(p_true, color="gray", ls=":", label=f"true p = {p_true}")
plt.xlabel("p"); plt.ylabel("log-likelihood"); plt.title("Bernoulli log-likelihood")
plt.legend(); plt.show()

##Example 2: Normal

Xᵢ ~ Normal(μ, σ²), both unknown.  ℓ(μ, σ²) = −(n/2)·log(2πσ²) − ∑(xᵢ − μ)² / (2σ²)

Setting both partial derivatives to 0:  μ̂ = x̄  σ̂² = ∑(xᵢ − x̄)² / n

σ̂² divides by n, not n − 1. Below, the formulas are checked against a numerical optimizer (used when no formula exists).

In [ ]:
mu_true, sigma_true = 5.0, 2.0
x_norm = rng.normal(mu_true, sigma_true, size=40)
n = len(x_norm)

mu_hat = x_norm.mean()
sigma2_hat = np.mean((x_norm - mu_hat) ** 2)

#minimize the negative log-likelihood &
#optimize log(sigma) so sigma stays positive
def normal_nll(params, x):
    mu, log_sigma = params
    return -np.sum(stats.norm.logpdf(x, loc=mu, scale=np.exp(log_sigma)))

res = optimize.minimize(normal_nll, x0=[0.0, 0.0], args=(x_norm,))
mu_num, sigma2_num = res.x[0], np.exp(res.x[1]) ** 2

print(f"{'':16s}{'mu':>10s}{'sigma^2':>10s}")
print(f"{'True':16s}{mu_true:>10.4f}{sigma_true**2:>10.4f}")
print(f"{'Formula':16s}{mu_hat:>10.4f}{sigma2_hat:>10.4f}")
print(f"{'Optimizer':16s}{mu_num:>10.4f}{sigma2_num:>10.4f}")
print(f"\nnp.var ddof=0 (divide by n):   {np.var(x_norm, ddof=0):.4f}")
print(f"np.var ddof=1 (divide by n-1): {np.var(x_norm, ddof=1):.4f}")

In [ ]:
#log-likelihood surface, the MLE is at the peak
mu_grid = np.linspace(mu_hat - 2, mu_hat + 2, 200)
s2_grid = np.linspace(1, 9, 200)
M, S2 = np.meshgrid(mu_grid, s2_grid)
LL = -n / 2 * np.log(2 * np.pi * S2) - ((x_norm[:, None, None] - M) ** 2).sum(axis=0) / (2 * S2)

plt.figure(figsize=(6.5, 5))
cs = plt.contour(M, S2, LL, levels=30, cmap="viridis")
plt.colorbar(cs, label="log-likelihood")
plt.plot(mu_hat, sigma2_hat, "r*", ms=15, label="MLE")
plt.plot(mu_true, sigma_true**2, "kx", ms=10, label="true value")
plt.xlabel("μ"); plt.ylabel("σ²"); plt.title("Normal log-likelihood contours")
plt.legend(); plt.show()

##Example 3: Exponential

Xᵢ ~ Exponential(λ), f(x) = λe^(−λx), e.g. time between customers.
ℓ(λ) = n·log(λ) − λ·∑xᵢ
ℓ'(λ) = n/λ − ∑xᵢ = 0   →   λ̂ = 1/x̄

Invariance: if θ̂ is the MLE of θ, then g(θ̂) is the MLE of g(θ). So the MLE of the mean wait 1/λ is 1/λ̂ = x̄.

In [ ]:
lam_true = 0.5
x_exp = rng.exponential(scale=1 / lam_true, size=60)

lam_hat = 1 / x_exp.mean()
res = optimize.minimize_scalar(lambda lam: -(len(x_exp) * np.log(lam) - lam * x_exp.sum()),
                               bounds=(1e-6, 10), method="bounded")

print(f"True lambda             = {lam_true}")
print(f"Formula 1/xbar          = {lam_hat:.4f}")
print(f"Optimizer               = {res.x:.4f}")
print(f"MLE of mean wait 1/lam  = {1/lam_hat:.4f}  (xbar = {x_exp.mean():.4f})")

xs = np.linspace(0, x_exp.max(), 300)
plt.hist(x_exp, bins=15, density=True, alpha=0.5, label="data")
plt.plot(xs, lam_hat * np.exp(-lam_hat * xs), "r", lw=2, label=f"MLE fit, λ̂ = {lam_hat:.2f}")
plt.plot(xs, lam_true * np.exp(-lam_true * xs), "k:", lw=2, label=f"true, λ = {lam_true}")
plt.xlabel("waiting time"); plt.ylabel("density"); plt.title("Exponential MLE fit")
plt.legend(); plt.show()

##Properties of the MLE (checked by simulation)

###1. Bias
For the normal variance: average of σ̂² = ((n − 1)/n)·σ². So σ̂² underestimates σ². The bias goes to 0 as n grows. Dividing by n − 1 removes it.

In [ ]:
sigma2, reps = 4.0, 20000
for n in [5, 20, 100]:
    samples = rng.normal(0, np.sqrt(sigma2), size=(reps, n))
    print(f"n = {n:3d} | avg MLE = {samples.var(axis=1, ddof=0).mean():.3f}"
          f"  theory = {(n-1)/n*sigma2:.3f}"
          f" | avg s^2 = {samples.var(axis=1, ddof=1).mean():.3f}  (true = {sigma2})")

###2. Consistency
As n → ∞, θ̂ → θ. The plot tracks λ̂ = 1/x̄ as observations are added.

In [ ]:
big_sample = rng.exponential(scale=1 / lam_true, size=5000)
n_vals = np.arange(1, len(big_sample) + 1)
running_mle = n_vals / np.cumsum(big_sample)

plt.semilogx(n_vals, running_mle, label="λ̂ after n observations")
plt.axhline(lam_true, color="red", ls="--", label=f"true λ = {lam_true}")
plt.ylim(0, 1.5)
plt.xlabel("n (log scale)"); plt.ylabel("estimate"); plt.title("Consistency of the MLE")
plt.legend(); plt.show()

###3. Asymptotic normality
For large n: θ̂ ≈ Normal(θ, 1 / (n·I(θ))) where the Fisher information is I(θ) = −E[second derivative of log f(X; θ)].

Exponential: second derivative of log f = −1/λ², so I(λ) = 1/λ² and λ̂ ≈ Normal(λ, λ²/n)

This also gives an approximate 95% CI: θ̂ ± 1.96 / √(n·I(θ̂)).

In [ ]:
n, reps = 100, 10000
lam_hats = 1 / rng.exponential(scale=1 / lam_true, size=(reps, n)).mean(axis=1)
theory_sd = lam_true / np.sqrt(n)

print(f"Mean of simulated MLEs = {lam_hats.mean():.4f}  (true = {lam_true})")
print(f"SD of simulated MLEs   = {lam_hats.std():.4f}  (theory = {theory_sd:.4f})")

grid = np.linspace(lam_hats.min(), lam_hats.max(), 300)
plt.hist(lam_hats, bins=50, density=True, alpha=0.5, label="simulated λ̂")
plt.plot(grid, stats.norm.pdf(grid, lam_true, theory_sd), "r", lw=2, label="Normal(λ, λ²/n)")
plt.xlabel("λ̂"); plt.ylabel("density"); plt.title("Asymptotic normality (n = 100)")
plt.legend(); plt.show()

# 95% CI for lambda from the Example 3 data
se = lam_hat / np.sqrt(len(x_exp))
print(f"\nExample 3: lambda_hat = {lam_hat:.3f}, 95% CI = ({lam_hat - 1.96*se:.3f}, {lam_hat + 1.96*se:.3f})")

#2.4.2 Linear Regression

Model: y = Xβ + ε,   ε ~ Normal(0, σ²I)

X is the design matrix (first column all 1s for the intercept).

Each yᵢ ~ Normal(row i of X · β, σ²), so ℓ(β, σ²) = −(n/2)·log(2πσ²) − RSS(β) / (2σ²),   RSS(β) = ‖y − Xβ‖²

β only appears in −RSS, so maximizing ℓ = minimizing RSS. With normal errors, the MLE of β is the least-squares estimate.

Setting the gradient of RSS to 0 gives the normal equations: XᵀX β̂ = Xᵀy   →   β̂ = (XᵀX)⁻¹Xᵀy

Variance: MLE σ̂² = RSS/n; unbiased version = RSS/(n − p − 1).

##Example 4: Simple linear regression, three methods

Data from y = 2 + 1.5x + noise. β is estimated by (1) the normal equations, (2) np.linalg.lstsq, (3) maximizing the likelihood numerically. All three should agree.

In [ ]:
n = 60
beta_true = np.array([2.0, 1.5])
sigma_true = 1.5

x = rng.uniform(0, 10, size=n)
X = np.column_stack([np.ones(n), x])
y = X @ beta_true + rng.normal(0, sigma_true, size=n)

beta_ne = np.linalg.solve(X.T @ X, X.T @ y)          #1 normal equations
beta_ls, *_ = np.linalg.lstsq(X, y, rcond=None)      #2 least squares

def regression_nll(params, X, y):                    #3 numerical MLE
    beta, log_sigma = params[:-1], params[-1]
    return -np.sum(stats.norm.logpdf(y, loc=X @ beta, scale=np.exp(log_sigma)))

res = optimize.minimize(regression_nll, x0=np.zeros(3), args=(X, y))
beta_mle, sigma2_mle_num = res.x[:2], np.exp(res.x[2]) ** 2

print(f"{'Method':20s}{'intercept':>12s}{'slope':>10s}")
print(f"{'True':20s}{beta_true[0]:>12.4f}{beta_true[1]:>10.4f}")
print(f"{'Normal equations':20s}{beta_ne[0]:>12.4f}{beta_ne[1]:>10.4f}")
print(f"{'lstsq':20s}{beta_ls[0]:>12.4f}{beta_ls[1]:>10.4f}")
print(f"{'Numerical MLE':20s}{beta_mle[0]:>12.4f}{beta_mle[1]:>10.4f}")

rss = np.sum((y - X @ beta_ne) ** 2)
print(f"\nsigma^2: RSS/n = {rss/n:.4f}, optimizer = {sigma2_mle_num:.4f}, "
      f"RSS/(n-2) = {rss/(n-2):.4f}, true = {sigma_true**2}")

In [ ]:
y_hat = X @ beta_ne
resid = y - y_hat
order = np.argsort(x)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x, y, alpha=0.7, label="data")
ax[0].plot(x[order], y_hat[order], "r", lw=2, label=f"fit: {beta_ne[0]:.2f} + {beta_ne[1]:.2f}x")
ax[0].plot(x[order], (X @ beta_true)[order], "k:", lw=2, label="true line")
for xi, yi, yh in zip(x, y, y_hat):
    ax[0].plot([xi, xi], [yi, yh], color="gray", lw=0.6)
ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].set_title("Least-squares fit (gray = residuals)")
ax[0].legend()

ax[1].scatter(y_hat, resid, alpha=0.7)
ax[1].axhline(0, color="red", ls="--")
ax[1].set_xlabel("fitted value"); ax[1].set_ylabel("residual"); ax[1].set_title("Residuals vs fitted")
plt.tight_layout(); plt.show()

##Geometry: the hat matrix

    ŷ = Hy,   H = X(XᵀX)⁻¹Xᵀ

H projects y onto the column space of X. Properties:
- Hᵀ = H and H² = H
- trace(H) = number of parameters
- Residuals are orthogonal to the columns of X: Xᵀe = 0 (the normal equations). With an intercept, ∑eᵢ = 0.

Values around 1e-13 are 0 up to rounding.

In [ ]:
H = X @ np.linalg.solve(X.T @ X, X.T)

print("max|H - H^T|  =", np.abs(H - H.T).max())
print("max|H@H - H|  =", np.abs(H @ H - H).max())
print("trace(H)      =", round(np.trace(H), 10))
print("H @ y = y_hat:", np.allclose(H @ y, y_hat))
print("X^T e         =", X.T @ resid)
print("sum of resid  =", resid.sum())

## R²
TSS = ESS + RSS   (total = explained + residual)
  R² = 1 − RSS/TSS

With one feature, R² = r² (squared correlation of x and y).

In [ ]:
tss = np.sum((y - y.mean()) ** 2)
ess = np.sum((y_hat - y.mean()) ** 2)
print(f"TSS = {tss:.3f}, ESS + RSS = {ess + rss:.3f}")
print(f"R^2 = {1 - rss/tss:.4f}, r^2 = {np.corrcoef(x, y)[0, 1]**2:.4f}")

##Example 5: Polynomial regression

"Linear" means linear in β. A cubic  y = β₀ + β₁x + β₂x² + β₃x³ + ε uses the same normal equations with columns 1, x, x², x³ (multiple regression, p = 3).

In [ ]:
n = 80
xp = rng.uniform(-3, 3, size=n)
beta_poly_true = np.array([1.0, -2.0, 0.5, 0.4])
Xp = np.column_stack([xp ** k for k in range(4)])
yp = Xp @ beta_poly_true + rng.normal(0, 1.0, size=n)

beta_poly = np.linalg.solve(Xp.T @ Xp, Xp.T @ yp)
X1 = Xp[:, :2]
beta_line = np.linalg.solve(X1.T @ X1, X1.T @ yp)

def r_squared(Xm, b, yv):
    return 1 - np.sum((yv - Xm @ b) ** 2) / np.sum((yv - yv.mean()) ** 2)

print("True beta:     ", beta_poly_true)
print("Estimated beta:", np.round(beta_poly, 3))
print(f"R^2 line = {r_squared(X1, beta_line, yp):.3f}, R^2 cubic = {r_squared(Xp, beta_poly, yp):.3f}")

grid = np.linspace(-3, 3, 300)
Xg = np.column_stack([grid ** k for k in range(4)])
plt.scatter(xp, yp, alpha=0.6, label="data")
plt.plot(grid, Xg @ beta_poly, "r", lw=2, label="cubic fit")
plt.plot(grid, Xg[:, :2] @ beta_line, "g--", lw=2, label="line fit")
plt.xlabel("x"); plt.ylabel("y"); plt.title("Polynomial regression")
plt.legend(); plt.show()

## Sampling distribution of β̂

β̂ is linear in the normal vector y, so β̂ ~ Normal(β, σ²(XᵀX)⁻¹)

β̂ is unbiased; standard errors = square roots of the diagonal of σ²(XᵀX)⁻¹. Checked by refitting Example 4 with 10,000 new noise draws.

In [ ]:
reps = 10000
Y_sim = (X @ beta_true)[None, :] + rng.normal(0, sigma_true, size=(reps, len(y)))
B_sim = np.linalg.solve(X.T @ X, X.T @ Y_sim.T).T

cov_theory = sigma_true ** 2 * np.linalg.inv(X.T @ X)
print("Mean of beta_hat:", np.round(B_sim.mean(axis=0), 4), " true:", beta_true)
print("Simulated SDs:   ", np.round(B_sim.std(axis=0), 4))
print("Theory SDs:      ", np.round(np.sqrt(np.diag(cov_theory)), 4))

grid = np.linspace(B_sim[:, 1].min(), B_sim[:, 1].max(), 300)
plt.hist(B_sim[:, 1], bins=60, density=True, alpha=0.5, label="simulated slopes")
plt.plot(grid, stats.norm.pdf(grid, beta_true[1], np.sqrt(cov_theory[1, 1])), "r", lw=2,
         label="theory")
plt.xlabel("slope estimate"); plt.title("Sampling distribution of the slope")
plt.legend(); plt.show()

##Example 6: Diabetes data (multiple regression)

442 patients, 10 standardized features, response = disease progression after one year. Fit with the normal equations and compare to scikit-learn. t = β̂ⱼ / SE(β̂ⱼ); |t| > 2 suggests the feature matters.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

data = load_diabetes()
Xd_raw, yd = data.data, data.target
n, p = Xd_raw.shape
Xd = np.column_stack([np.ones(n), Xd_raw])

beta_d = np.linalg.solve(Xd.T @ Xd, Xd.T @ yd)
rss_d = np.sum((yd - Xd @ beta_d) ** 2)
sigma2_unb = rss_d / (n - p - 1)
se_d = np.sqrt(np.diag(sigma2_unb * np.linalg.inv(Xd.T @ Xd)))
t_stats = beta_d / se_d

sk = LinearRegression().fit(Xd_raw, yd)
sk_coefs = np.concatenate([[sk.intercept_], sk.coef_])

names = ["intercept"] + list(data.feature_names)
print(f"{'term':>10s}{'beta_hat':>11s}{'sklearn':>11s}{'SE':>9s}{'t':>8s}")
for nm, b, bs, se, t in zip(names, beta_d, sk_coefs, se_d, t_stats):
    print(f"{nm:>10s}{b:>11.2f}{bs:>11.2f}{se:>9.2f}{t:>8.2f}")

print(f"\nMatches sklearn: {np.allclose(beta_d, sk_coefs)}")
print(f"R^2 = {1 - rss_d / np.sum((yd - yd.mean())**2):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(Xd @ beta_d, yd, alpha=0.5)
lims = [yd.min(), yd.max()]
ax[0].plot(lims, lims, "r--")
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("actual"); ax[0].set_title("Actual vs predicted")

ax[1].barh(names[1:], t_stats[1:])
ax[1].axvline(2, color="red", ls=":"); ax[1].axvline(-2, color="red", ls=":")
ax[1].set_xlabel("t-statistic"); ax[1].set_title("|t| > 2 ≈ significant")
plt.tight_layout(); plt.show()

# Summary

2.4.1
- MLE maximizes ℓ(θ) = ∑ log f(xᵢ; θ).
- Bernoulli p̂ = s/n; Normal μ̂ = x̄, σ̂² = ∑(xᵢ − x̄)²/n; Exponential λ̂ = 1/x̄.
- The MLE can be biased, but it is consistent, approximately Normal(θ, 1/(n·I(θ))), and invariant.

2.4.2
- With normal errors, MLE of β = least squares: XᵀXβ̂ = Xᵀy.
- ŷ = Hy is a projection; residuals are orthogonal to the columns of X.
- β̂ ~ Normal(β, σ²(XᵀX)⁻¹) gives standard errors and t-statistics; R² = 1 − RSS/TSS.